In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "ACQ_G.xpt"
csv_file = DATA_DIR / "ACQ_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve the completely untouched pandas-decoded XPT.
#
source_df = df.copy()


print("=" * 80)
print("NHANES 2011-2012 ACQ_G XPT -> CSV")
print("=" * 80)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "ACD011A",
    "ACD011B",
    "ACD011C",
    "ACD040",
    "ACD110"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "ACQ_G column names/order differ from expected.\n"
        f"Expected: {EXPECTED_COLUMNS}\n"
        f"Actual:   {df.columns.tolist()}"
    )


if len(df) != 6833:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 6:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 6,833 rows x 6 columns."
)


# ============================================================
# 4. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# Attached ACQ_G.xpt:
#
#     tiny artifact count = 0
#
# Therefore NO correction is expected.
#
# If a local/future copy contains this value,
# STOP rather than replacing it automatically.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")


print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


print(
    "Tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        "\nTiny values by variable:"
    )

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in ACQ_G. Do not automatically "
        "replace them with zero."
    )


if true_zero_total != 0:

    raise ValueError(
        "STOP: Unexpected numeric zero values "
        "found in ACQ_G."
    )


print(
    "PASS: No tiny XPORT artifacts."
)

print(
    "PASS: No numeric zeros."
)


# ============================================================
# 5. CHECK FOR GENUINE DECIMAL VARIABLES
# ============================================================

#
# IMPORTANT:
#
# Always do this BEFORE Int64 conversion.
#
# Attached ACQ_G:
#
#     genuine fractional variables = NONE
#
# All variables are identifiers or coded questionnaire
# responses.
#
# If a decimal unexpectedly appears, STOP.
# DO NOT round it.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:

        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    fractional_count = int(
        fractional_mask.sum()
    )


    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )


    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values unexpectedly "
        "found in ACQ_G. They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional variables."
)


# ============================================================
# 6. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 7. FORMAT ALL PROVEN WHOLE-NUMBER VARIABLES
# ============================================================

#
# ACQ_G has NO genuine decimal variables.
#
# All 6 columns are therefore safe to store as nullable Int64.
#
# IMPORTANT:
#
# NO .round() is used.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal "
                "values and cannot safely be Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 6 ACQ_G variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 8. HELPER - CDC FREQUENCY CHECK
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )


    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )


    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n"
            f"Expected: {expected_values}\n"
            f"Actual:   {actual_values}"
        )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing}\n"
            f"Actual:   {actual_missing}"
        )


    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 9. CDC CHECK - ACD011A
# ============================================================

#
# Speak English at home
#
# CDC:
#
# 1  English       4,279
# 77 Refused           1
# 99 Don't know        0
# Missing          2,553
#

check_frequency(
    "ACD011A",
    {
        1: 4279,
        77: 1
    },
    2553
)


# ============================================================
# 10. CDC CHECK - ACD011B
# ============================================================

#
# Speak Spanish at home
#
# CDC:
#
# 8 Spanish = 19
# Missing   = 6,814
#

check_frequency(
    "ACD011B",
    {
        8: 19
    },
    6814
)


# ============================================================
# 11. CDC CHECK - ACD011C
# ============================================================

#
# Speak other language at home
#
# CDC:
#
# 9 Other  = 155
# Missing  = 6,678
#

check_frequency(
    "ACD011C",
    {
        9: 155
    },
    6678
)


# ============================================================
# 12. CDC CHECK - ACD040
# ============================================================

#
# Language spoken at home - Hispanic participants
#
# CDC:
#
# 1 = Only Spanish                  543
# 2 = More Spanish than English     203
# 3 = Both equally                  258
# 4 = More English than Spanish     250
# 5 = Only English                  221
# 7 = Refused                         0
# 9 = Don't know                      5
# Missing                         5,353
#

check_frequency(
    "ACD040",
    {
        1: 543,
        2: 203,
        3: 258,
        4: 250,
        5: 221,
        9: 5
    },
    5353
)


# ============================================================
# 13. CDC CHECK - ACD110
# ============================================================

#
# Language spoken at home - single-race
# non-Hispanic Asian participants
#
# CDC:
#
# 1 = Only non-English language          455
# 2 = More non-English than English       66
# 3 = Both equally                        65
# 4 = More English than non-English       97
# 5 = Only English                       289
# 7 = Refused                              0
# 9 = Don't know                           0
# Missing                              5,861
#

check_frequency(
    "ACD110",
    {
        1: 455,
        2: 66,
        3: 65,
        4: 97,
        5: 289
    },
    5861
)


# ============================================================
# 14. INTERNAL NON-MISSING COUNT CHECKS
# ============================================================

#
# These totals also provide a useful consistency check.
#

EXPECTED_NONMISSING = {

    "ACD011A": 4280,

    "ACD011B": 19,

    "ACD011C": 155,

    "ACD040": 1480,

    "ACD110": 972
}


print("\n--- NON-MISSING COUNT CHECK ---")


for col, expected_count in EXPECTED_NONMISSING.items():

    actual_count = int(
        df[col]
        .notna()
        .sum()
    )


    print(
        f"{col}: "
        f"{actual_count:,}"
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: expected "
            f"{expected_count:,} non-missing values, "
            f"found {actual_count:,}."
        )


print(
    "PASS: All non-missing counts match CDC."
)


# ============================================================
# 15. CHECK ALLOWED RELEASED VALUES
# ============================================================

#
# This is an additional safeguard against accidental
# corruption or unexpected codes.
#

ALLOWED_VALUES = {

    "ACD011A": {
        1,
        77
    },

    "ACD011B": {
        8
    },

    "ACD011C": {
        9
    },

    "ACD040": {
        1,
        2,
        3,
        4,
        5,
        9
    },

    "ACD110": {
        1,
        2,
        3,
        4,
        5
    }
}


print("\n--- ALLOWED CODE CHECK ---")


for col, allowed in ALLOWED_VALUES.items():

    observed = set(
        df[col]
        .dropna()
        .astype(int)
        .unique()
        .tolist()
    )


    unexpected = (
        observed
        -
        allowed
    )


    if unexpected:

        raise ValueError(
            f"{col} contains unexpected codes: "
            f"{sorted(unexpected)}"
        )


    print(
        f"PASS: {col} contains only expected "
        "released values."
    )


# ============================================================
# 16. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# ACQ_G requires ZERO numeric corrections.
#
# Therefore every numeric value must remain exactly
# equal to the source XPT.
#
# Only its CSV textual representation changes:
#
#     SEQN
#     62161.0 -> 62161
#
#     ACD040
#     3.0 -> 3
#
# etc.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 6 variables preserve "
    "source XPT values exactly."
)


# ============================================================
# 17. EXPORT CSV
# ============================================================

#
# No rounding.
# No float_format.
#
# Missing values become blank.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 18. RE-READ EXPORTED CSV
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed after export."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 19. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- EXACT XPT -> CSV FIDELITY ---")


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV differences found:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 6 CSV columns round-trip "
    "to exact source XPT numeric values."
)


# ============================================================
# 20. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 21. READ RAW CSV AS TEXT
# ============================================================

#
# Numeric fidelity was checked above.
#
# Now inspect exact CSV formatting.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


# ============================================================
# 22. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    numeric_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        numeric_values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts."
)


# ============================================================
# 23. RAW CSV .0 FORMAT CHECK
# ============================================================

#
# ACQ_G has NO genuine decimal variables.
#
# Therefore none of these fields should end with ".0".
#
# This global ACQ_G check is safe specifically because
# every variable in this file is a proven whole-number
# identifier/code.
#
# DO NOT apply this logic globally to Dietary files
# or Labs/Exam files containing genuine decimals.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 formatting."
)


# ============================================================
# 24. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 25. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains .0."
    )


print(
    "PASS: Final CSV SEQN valid "
    "and contains no .0."
)


# ============================================================
# 26. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 80
)


print(
    "ACQ_G CONVERSION COMPLETE"
)


print(
    "=" * 80
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    tiny_total
)


print(
    "Validated zero restorations: NONE"
)


print(
    "Genuine decimal variables: NONE"
)


print(
    "Whole-number variables:",
    len(INTEGER_COLUMNS)
)


print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)


print(
    "Missing-position differences: "
    "0 expected"
)


print(
    "Tiny values remaining: "
    "0 expected"
)


print(
    "Unnecessary .0: "
    "0 expected"
)


print(
    "Scientific notation: "
    "0 expected"
)

NHANES 2011-2012 ACQ_G XPT -> CSV
Rows: 6,833
Columns: 6
PASS: Structure = 6,833 rows x 6 columns.

--- ORIGINAL XPT ZERO CHECK ---
Ordinary numeric zeros: 0
Tiny XPORT artifacts: 0
PASS: No tiny XPORT artifacts.
PASS: No numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
ACD011A: 0 fractional observations
ACD011B: 0 fractional observations
ACD011C: 0 fractional observations
ACD040: 0 fractional observations
ACD110: 0 fractional observations
PASS: No genuine fractional variables.

--- SEQN ---
Minimum: 62161
Maximum: 71916
Missing: 0
Duplicates: 0
PASS: SEQN validated.

PASS: All 6 ACQ_G variables converted to nullable Int64 without rounding.
PASS: ACD011A matches CDC.
PASS: ACD011B matches CDC.
PASS: ACD011C matches CDC.
PASS: ACD040 matches CDC.
PASS: ACD110 matches CDC.

--- NON-MISSING COUNT CHECK ---
ACD011A: 4,280
ACD011B: 19
ACD011C: 155
ACD040: 1,480
ACD110: 972
PASS: All non-missing counts match CDC.

--- ALLOWED CODE CHECK ---
PASS: ACD011A contain